# 05. 그룹 기반 split과 통합 manifest

같은 환자/검사 그룹이 서로 다른 split에 들어가지 않도록 먼저 그룹 단위로 train/validation/test를 배정합니다. 예외 검토가 끝나지 않은 케이스도 split은 고정하되 `HOLD_REVIEW`로 표시하여 전처리에서 제외합니다.

In [1]:
from pathlib import Path
import hashlib
import json
import sys

import numpy as np
import pandas as pd

def find_project_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / 'project_config.json').is_file() and (candidate / 'ctqa_utils.py').is_file():
            return candidate
    raise FileNotFoundError('project_config.json이 있는 프로젝트 폴더를 여세요.')

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))
from ctqa_utils import as_bool

RESULTS = PROJECT_ROOT / 'results'
policy = json.loads((RESULTS / '04_integration_policy.json').read_text(encoding='utf-8'))
pairs = pd.read_csv(RESULTS / '00_pair_manifest.csv')
values = pd.read_csv(RESULTS / '02_value_label_checks.csv')
registry = pd.read_csv(RESULTS / '04_case_review_registry.csv').fillna('')
print('전체 pair:', len(pairs))

전체 pair: 267


## 1. 기술 검토 상태 병합

In [2]:
allowed = {
    'INTENSITY_SIGNED12': {'APPLY_SIGNED12', 'KEEP_CONFIRMED', 'EXCLUDE'},
    'AFFINE_MISMATCH': {'COPY_CT_AFFINE_TO_MASK', 'RESAMPLE_MASK_TO_CT', 'EXCLUDE'},
    'Z_SPACING_OUTLIER': {'KEEP_CONFIRMED', 'EXCLUDE'},
}
registry['approved_bool'] = registry.approved.map(as_bool)
registry['valid_action'] = registry.apply(lambda row: row.action in allowed[row.issue], axis=1)
registry['complete_review'] = (
    registry.approved_bool & registry.valid_action
    & registry.action.astype(str).str.strip().ne('')
    & registry.reviewer.astype(str).str.strip().ne('')
    & registry.rationale.astype(str).str.strip().ne('')
)
registry['approved_exclusion'] = registry.complete_review & registry.action.eq('EXCLUDE')
review_by_case = registry.groupby(['source', 'case_id']).agg(
    issue_count=('issue', 'count'),
    approved_issue_count=('complete_review', 'sum'),
    excluded=('approved_exclusion', 'max'),
).reset_index()
review_by_case['all_issues_approved'] = review_by_case.issue_count == review_by_case.approved_issue_count

manifest = pairs.merge(
    values[['source', 'case_id', 'empty_mask', 'positive_voxels']],
    on=['source', 'case_id'], how='left', validate='one_to_one',
).merge(review_by_case, on=['source', 'case_id'], how='left', validate='one_to_one')
manifest['positive_case'] = ~manifest.empty_mask.map(as_bool)
manifest['issue_count'] = manifest.issue_count.fillna(0).astype(int)
manifest['approved_issue_count'] = manifest.approved_issue_count.fillna(0).astype(int)
manifest['excluded'] = manifest.excluded.fillna(False).map(as_bool)
manifest['all_issues_approved'] = manifest.all_issues_approved.fillna(True).map(as_bool)
manifest['review_status'] = np.select(
    [manifest.excluded & manifest.all_issues_approved, ~manifest.all_issues_approved],
    ['EXCLUDED_APPROVED', 'HOLD_REVIEW'],
    default='READY',
)
manifest.groupby(['source', 'review_status']).size().rename('cases').reset_index()

,source,review_status,cases
0,BHSD,EXCLUDED_APPROVED,1
1,BHSD,READY,191
2,PhysioNet_CT_ICH,READY,75


## 2. source·양성 여부를 고려한 그룹 split

재현 가능한 난수로 그룹을 배정합니다. `group_id`가 같은 모든 케이스는 항상 동일 split을 받습니다. 공식 split 정보가 있다면 아래 override CSV에 그룹 단위로 기록하세요.

In [3]:
ratios = policy['split_ratios']
if not np.isclose(sum(ratios.values()), 1.0):
    raise ValueError('split_ratios 합은 1이어야 합니다.')
seed = int(policy['random_seed'])
active = manifest.loc[~manifest.excluded].copy()
groups = active.groupby(['source', 'group_id']).agg(
    cases=('case_id', 'count'),
    positive_group=('positive_case', 'max'),
).reset_index()
groups['stratum'] = groups.source + '_' + groups.positive_group.astype(int).astype(str)

assigned = []
for stratum, frame in groups.groupby('stratum', sort=True):
    local_seed = seed + int(hashlib.sha256(stratum.encode()).hexdigest()[:8], 16)
    shuffled = frame.iloc[np.random.default_rng(local_seed).permutation(len(frame))].copy()
    n = len(shuffled)
    n_test = int(round(n * ratios['test']))
    n_validation = int(round(n * ratios['validation']))
    if n >= 3:
        n_test = max(1, n_test)
        n_validation = max(1, n_validation)
    if n_test + n_validation >= n:
        n_validation = max(0, n - n_test - 1)
    labels = np.array(['train'] * n, dtype=object)
    labels[:n_test] = 'test'
    labels[n_test:n_test + n_validation] = 'validation'
    shuffled['split'] = labels
    assigned.append(shuffled)
group_split = pd.concat(assigned, ignore_index=True)

override_path = RESULTS / '05_split_overrides.csv'
if not override_path.is_file():
    pd.DataFrame(columns=['source', 'group_id', 'split', 'reason']).to_csv(override_path, index=False, encoding='utf-8-sig')
overrides = pd.read_csv(override_path).dropna(how='all')
if len(overrides):
    if not set(overrides.split).issubset({'train', 'validation', 'test'}):
        raise ValueError('override split은 train/validation/test만 허용됩니다.')
    if overrides.duplicated(['source', 'group_id']).any():
        raise ValueError('override에 중복 group이 있습니다.')
    override_map = overrides.set_index(['source', 'group_id']).split.to_dict()
    group_split['split'] = group_split.apply(lambda r: override_map.get((r.source, r.group_id), r.split), axis=1)
group_split[['source', 'group_id', 'split']].head()

,source,group_id,split
0,BHSD,ID_f4795940,test
1,BHSD,ID_6a6976ae,test
2,BHSD,ID_eac5718d,test
3,BHSD,ID_dd3d7720,test
4,BHSD,ID_69b19057,test


## 3. 최종 manifest 저장과 누수 검사

In [4]:
manifest = manifest.merge(
    group_split[['source', 'group_id', 'split']],
    on=['source', 'group_id'], how='left', validate='many_to_one',
)
manifest.loc[manifest.excluded, 'split'] = 'excluded'
manifest['eligible_for_preprocessing'] = manifest.review_status.eq('READY')
manifest['task'] = policy['task']
manifest['binary_label_rule'] = policy['binary_label_rule']
manifest = manifest.sort_values(['source', 'split', 'group_id', 'case_id']).reset_index(drop=True)

overlap = (
    manifest.loc[manifest.split.isin(['train', 'validation', 'test'])]
    .groupby(['source', 'group_id']).split.nunique()
)
overlap = overlap[overlap > 1]
if len(overlap):
    raise RuntimeError(f'그룹 split 누수 {len(overlap)}건')

output_path = RESULTS / '05_integration_manifest.csv'
manifest.to_csv(output_path, index=False, encoding='utf-8-sig')
summary = manifest.groupby(['source', 'split']).agg(
    cases=('case_id', 'count'),
    groups=('group_id', 'nunique'),
    positive_cases=('positive_case', 'sum'),
    ready_cases=('eligible_for_preprocessing', 'sum'),
).reset_index()
display(summary)
print('group overlap:', len(overlap))
print('통합 manifest:', output_path)

,source,split,cases,groups,positive_cases,ready_cases
0,BHSD,excluded,1,1,1,0
1,BHSD,test,28,28,28,28
2,BHSD,train,135,134,135,135
3,BHSD,validation,28,28,28,28
4,PhysioNet_CT_ICH,test,12,12,7,12
5,PhysioNet_CT_ICH,train,51,51,30,51
6,PhysioNet_CT_ICH,validation,12,12,7,12


group overlap: 0
통합 manifest: c:\workspace\hodgepodge\hy\데이터품질검증 및 통합\results\05_integration_manifest.csv
